# Stage 1–6 demo using your developed methods

This notebook uses your existing modules where they are runnable: `pdfParser.py`, `mdParser.py`, `reliability_summarizer.py`, `equipment_ID_extractor.py`, and `chroma_store.py`. It prefers your implementations first and only falls back when a runtime dependency like Marker or Ollama is unavailable.


In [ ]:
import sys
import os
rca_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(rca_root)

dackar_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(dackar_root)

In [2]:
from pathlib import Path
import pandas as pd
from stage1_6_existing_methods_helpers import run_many, maybe_upsert_with_chroma

PDFS = [
    '../examples/example_CR_2026_00123.pdf',
    # '../examples/example_ECA_2026_0007.pdf',
    # '../examples/example_SOP_AFW_P101A.pdf',
    # '../examples/example_WO_2026_04567.pdf',
]

OUTPUT_ROOT = './1-6pipeline_demo_existing_methods'

In [3]:
results = run_many(PDFS, output_root=OUTPUT_ROOT, model=None)
len(results)

2026-03-25 15:35:06 | INFO | pdfParser | Rendering PDF to Markdown via Marker: ../examples/example_CR_2026_00123.pdf
2026-03-25 15:35:08,839 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead
Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00, 14.89it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:02<00:00,  2.04s/it]
Detecting bboxes: 0it [00:00, ?it/s]
2026-03-25 15:35:20 | INFO | pdfParser | Extracting tables via pdfplumber: ../examples/example_CR_2026_00123.pdf
2026-03-25 15:35:20 | INFO | pdfParser | Parsing complete for: ../examples/example_CR_2026_00123.pdf
2026-03-25 15:35:20,492 | INFO | mdParser | Reading Markdown text from: 1-6pipeline_demo_existing_methods/example_CR_2026_00123/stage1/ad61d26ecf6e/text/example_CR_2026_00123.md
2026-03-25 15:35:20,493 | INFO | mdParser | Assigning tables to sections with heuristic matching.
2026-03-25 15:35:20,494 | INFO | mdParser | R

1

In [4]:
rows = []
for r in results:
    rows.append({
        'pdf': Path(r.pdf_path).name,
        'doc_type': r.doc_type,
        'enriched_jsonl_path': r.enriched_jsonl_path,
        'processed_record_count': len(r.processed_records),
    })
pd.DataFrame(rows)


,pdf,doc_type,enriched_jsonl_path,processed_record_count
0,example_CR_2026_00123.pdf,CR,1-6pipeline_demo_existing_methods/example_CR_2...,9


In [5]:
# Inspect one processed_text_record
sample = results[0].processed_records[0]
sample.keys(), sample['metadata'].keys()


(dict_keys(['record_id', 'doc_id', 'doc_type', 'chunk_index', 'embedding_text', 'metadata', 'provenance', 'enrichment']),
 dict_keys(['doc_type', 'authority_level', 'section_role', 'page_start', 'page_end', 'equipment_ids', 'system_names', 'component_names', 'mechanisms', 'failure_outcomes', 'maintenance_actions', 'surveillance_actions', 'tools_methods', 'properties_or_limits', 'has_causal_language', 'has_diagnostics', 'has_maintenance_action', 'has_surveillance_action', 'has_failure_signal', 'has_explicit_causal_statement', 'has_condition_state', 'has_procedural_deviation']))

In [6]:
sample


{'record_id': 'ad61d26ecf6e::0',
 'doc_id': 'ad61d26ecf6e',
 'doc_type': 'CR',
 'chunk_index': 0,
 'embedding_text': "SCOPE: | Field | Value | Field | Value | |----------|-----------------------------|----------------|--------------------| | CR ID | CR-2026-00123 | Date Initiated | 2026-02-06 | | System | AFW | Equipment | P-101A | | Location | Turbine Building Elev. 468' | Initiator | Operator (Example) |\n\nAFW Train B remained available. No safety injection or reactor trip occurred. This CR is categorized as low safety\nSYSTEMS: \nEQUIPMENT: P-101A, VB-101A, FT-1102, PT-1102\nCOMPONENTS: P-101A, VB-101A, FT-1102, PT-1102\nSYMPTOMS/OUTCOMES: \nMECHANISMS: \nDIAGNOSTICS: test\nACTIONS: \nNUMBERS/LIMITS: 101a, 101a, 5.0 mils\nKEYWORDS: field, value, field value, ----------, -----------------------------, ----------------, --------------------, cr-2026-00123, date, initiated, 2026-02-06, afw, safety, train, remained, available, injection, reactor, trip, occurred, categorized, low, signi

In [7]:
# Optional Chroma upsert if dependencies are installed and Ollama embeddings are available
enriched_paths = [r.enriched_jsonl_path for r in results]
maybe_upsert_with_chroma(enriched_paths, persist_directory = "./1-6pipeline_demo_existing_methods/chroma_store")


/Users/mandd/projects/DACKAR/src/dackar/RCA/storage/chroma_store.py:188: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  self.embedder = OllamaEmbeddings(base_url=self.ollama_base_url, model=self.embed_model)


{'status': 'ok',
 'counts': {'1-6pipeline_demo_existing_methods/example_CR_2026_00123/stage3_6/ad61d26ecf6e_chunks_enriched.jsonl': {'CR': 9}}}